In [1]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Keju@0573'
)
cursor = connection.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS aqi_pulse")
cursor.execute("USE aqi_pulse")
print("Connected to MySQL successfully!")
print("Database 'aqi_pulse' created and selected!")

Connected to MySQL successfully!
Database 'aqi_pulse' created and selected!


In [3]:
df = pd.read_csv('E:\\project_aqi_pulse\\data\\aqi_cleaned.csv')

print("CSV loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

cursor.execute("""
    CREATE TABLE IF NOT EXISTS aqi_data (
        id INT AUTO_INCREMENT PRIMARY KEY,
        `City` VARCHAR(50),
        `Date` DATE,
        `Month` INT,
        `Year` INT,
        `Season` VARCHAR(50),
        `PM2.5` FLOAT,
        `PM10` FLOAT,
        `NO` FLOAT,
        `NO2` FLOAT,
        `NH3` FLOAT,
        `CO` FLOAT,
        `SO2` FLOAT,
        `O3` FLOAT,
        `AQI` FLOAT,
        `AQI_Bucket` VARCHAR(50),
        `Dominant_Pollutant` VARCHAR(50),
        `PM2.5_Risk` VARCHAR(50)
    )
""")

print("Table 'aqi_data' created successfully!")

CSV loaded successfully!
Shape: (24850, 17)
Columns: ['City', 'Date', 'Month', 'Year', 'Season', 'PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 'AQI', 'AQI_Bucket', 'Dominant_Pollutant', 'PM2.5_Risk']
Table 'aqi_data' created successfully!


In [4]:
import numpy as np
df2 = df.copy()

df2 = df2.replace({np.nan: None})

insert_query = """
    INSERT INTO aqi_data (`City`, `Date`, `Month`, `Year`, `Season`, `PM2.5`, `PM10`, `NO`, `NO2`, `NH3`, `CO`, `SO2`, `O3`, `AQI`, 
    `AQI_Bucket`, `Dominant_Pollutant`, `PM2.5_Risk`    
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.execute("DELETE FROM aqi_data")

for index, row in df2.iterrows():
    values = (
        row['City'], row['Date'], row['Month'], row['Year'], row['Season'], row['PM2.5'], row['PM10'], row['NO'], 
        row['NO2'], row['NH3'], row['CO'], row['SO2'], row['O3'], row['AQI'], row['AQI_Bucket'], row['Dominant_Pollutant'], row['PM2.5_Risk']
    )
    cursor.execute(insert_query, values)

connection.commit()

print(f"Successfully inserted {len(df2)} rows into MySQL!")
print("Data is now live in aqi_pulse database!")

Successfully inserted 24850 rows into MySQL!
Data is now live in aqi_pulse database!


In [5]:
verify_query = "SELECT COUNT(*) FROM aqi_data"
cursor.execute(verify_query)

result = cursor.fetchone()

print(f"Total rows in MySQL: {result[0]}")

cursor.execute("SELECT * FROM aqi_data LIMIT 5")

rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

df_preview = pd.DataFrame(rows, columns=columns)

print("\nFirst 5 rows from MySQL:")
df_preview

Total rows in MySQL: 24850

First 5 rows from MySQL:


,id,City,Date,Month,Year,Season,PM2.5,PM10,NO,NO2,NH3,CO,SO2,O3,AQI,AQI_Bucket,Dominant_Pollutant,PM2.5_Risk
0,1,Ahmedabad,2015-01-29,1,2015,Winter,83.13,111.685,6.93,28.71,None,6.93,49.52,59.76,209.0,Poor,PM10,High
1,2,Ahmedabad,2015-01-30,1,2015,Winter,79.84,111.685,13.85,28.68,None,13.85,48.49,97.07,328.0,Very Poor,PM10,High
2,3,Ahmedabad,2015-01-31,1,2015,Winter,94.52,111.685,24.39,32.66,None,24.39,67.39,111.33,514.0,Severe,PM10,Very High
3,4,Ahmedabad,2015-02-01,2,2015,Winter,135.99,144.908,43.48,42.08,None,43.48,75.23,102.70,782.0,Severe,PM10,Very High
4,5,Ahmedabad,2015-02-02,2,2015,Winter,178.33,144.908,54.56,35.31,None,54.56,55.04,107.38,914.0,Severe,PM2.5,Very High
